In [0]:
-- Data Exploration

-- Question 1: What day of the wek is used for each week_date value?
SELECT  distinct date_format(week_date, 'EEEE')
 AS day_of_week
FROM clean_weekly_sales;

 -- Question 2: What range of week numbers are misisng from the dataset

with weeks as (SELECT explode(sequence(1, 52)) as week_number),

comp as (SELECT w.week_number,
cw.week_number as avail_week
FROM weeks w
LEFT JOIN clean_weekly_sales cw ON w.week_number = cw.week_number
GROUP BY w.week_number, cw.week_number)

SELECT count(*) as total_week_numbers_missing FROM comp
WHERE avail_week IS NULL;

-- Question 3: How many total transactions were there for each year in the dataset?
SELECT calendar_year,
sum(transactions) as total_transactions
FROM clean_weekly_sales
GROUP BY calendar_year
ORDER BY calendar_year;

-- Question 4: What is the total sales for each region for each month?

SELECT 
region,
month_number,
calendar_year,
sum(sales) as total_sales
FROM clean_weekly_sales
GROUP BY region, calendar_year, month_number
ORDER BY region, calendar_year, month_number, total_sales desc;

-- Question 5: What is the total count of transactions for each platform
SELECT platform,
count(transactions) as total_transactions 
FROM clean_weekly_sales
GROUP BY platform;

-- Question 6: What is the percentage of sales for Retail vs Shopify for each month?
WITH platform_sales AS (
  SELECT
    platform,
    month_number,
    calendar_year,
    SUM(sales) AS plat_sales
  FROM clean_weekly_sales
  GROUP BY platform, month_number, calendar_year
)

SELECT
  platform,
  month_number,
  calendar_year,
  plat_sales,
  SUM(plat_sales) OVER (PARTITION BY calendar_year, month_number) AS total_sales,
  ROUND(plat_sales * 100.0 / SUM(plat_sales) OVER (PARTITION BY calendar_year, month_number), 2) AS pct_sales
FROM platform_sales
ORDER BY calendar_year, month_number, platform;

-- Question 7: What is the percentage of sales by demographic for each year in the dataset?
WITH demo_data as (SELECT demographic,
calendar_year,
sum(sales) as demo_sales 
FROM clean_weekly_sales
GROUP BY demographic, calendar_year
ORDER BY calendar_year, demographic)

SELECT *,
SUM(demo_sales) OVER (PARTITION BY calendar_year ORDER BY calendar_year) as total_sales,
ROUND(demo_sales/total_sales*100, 2) as pct
FROM demo_data
ORDER BY calendar_year, demographic;

-- Question 8: Which age_band and demographic values contribute the most to Retail sales? 

SELECT age_band as demo_group,
sum(sales) as total_sales
FROM clean_weekly_sales
WHERE platform = 'Retail'
AND age_band <> 'unknown'
GROUP BY age_band


UNION ALL

SELECT demographic as demo_group,
sum(sales) as total_sales
FROM clean_weekly_sales
WHERE platform = 'Retail'
AND demographic <> 'unknown'
GROUP BY demographic

ORDER BY total_sales desc

-- Question 9: Can we use the avg_transaction column to find the average transaction size for each year for Retail vs Shopify? If not - how would you calculate it instead?

SELECT
  platform,
  calendar_year,
  ROUND(SUM(sales) / SUM(transactions), 2) AS avg_transaction_size
FROM clean_weekly_sales
GROUP BY platform, calendar_year
ORDER BY calendar_year, platform;







